# NB-301: Self-Attention Implementation

Implement self-attention mechanism from scratch.

## Learning Objectives
- Understand attention mechanism
- Implement scaled dot-product attention
- Build multi-head attention
- Visualize attention weights

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

print("Libraries imported successfully")

## 1. Scaled Dot-Product Attention

In [ ]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    Compute scaled dot-product attention.

    Args:
        query: (batch_size, seq_len, d_k)
        key: (batch_size, seq_len, d_k)
        value: (batch_size, seq_len, d_v)
        mask: Optional mask (batch_size, 1, seq_len)

    Returns:
        output: (batch_size, seq_len, d_v)
        attention_weights: (batch_size, seq_len, seq_len)
    """
    # TODO: Compute scores
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

    # TODO: Apply mask if provided
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # TODO: Apply softmax
    attention_weights = F.softmax(scores, dim=-1)

    # TODO: Multiply by values
    output = torch.matmul(attention_weights, value)

    return output, attention_weights

# Test
batch_size = 2
seq_len = 5
d_k = 4
d_v = 4

Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_v)

output, attn = scaled_dot_product_attention(Q, K, V)
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn.shape}")

## 2. Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # TODO: Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # TODO: Linear projections and reshape
        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # TODO: Apply attention
        attn_output, _ = scaled_dot_product_attention(Q, K, V, mask)

        # TODO: Concatenate and project
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.W_o(attn_output)

        return output

# Test
mha = MultiHeadAttention(d_model=512, num_heads=8)
x = torch.randn(2, 10, 512)  # (batch, seq_len, d_model)
output = mha(x, x, x)
print(f"MHA output shape: {output.shape}")

## 3. Visualize Attention Weights

In [ ]:
# Create sample attention weights
seq_len = 8
attn_weights = F.softmax(torch.randn(seq_len, seq_len), dim=-1)

# TODO: Plot attention
plt.figure(figsize=(8, 6))
plt.imshow(attn_weights.detach().numpy(), cmap='Blues')
plt.colorbar()
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Attention Weights')
plt.show()

## Exercise

1. Implement causal masking for decoder attention
2. Add dropout to attention weights
3. Compare single-head vs multi-head attention